# Project 2 — Environment and Cloud setup

Run this notebook on the Mac's Jupyter kernel for local preflight and Google Cloud setup. It is the only setup entry point. Spark/data analysis runs in `01_data_quality.ipynb` on the Dataproc Jupyter kernel.

The only permitted billing account is Education (ID `01DAD8-AE5F1A-E3910A`); the only permitted credit is `3LU63BPGY77JFDQQ`. The credit values below are the user's snapshot, not a live Billing API result. This notebook links the existing project to Education because the user explicitly requested that account. It does not create paid resources by default.

In [1]:
from pathlib import Path
import json
import subprocess

from IPython.display import display

PROJECT_ID = "new-cloud-256902"
BILLING_ACCOUNT_NAME = "Billing Account for Education"
BILLING_ACCOUNT_ID = "01DAD8-AE5F1A-E3910A"
ALLOWED_CREDIT_ID = "3LU63BPGY77JFDQQ"
REGION = "asia-southeast1"
CLUSTER = "dsa5208-rainfall"
BUCKET = ""  # Fill only after checking global availability and estimating storage cost.
DATA_ROOT = Path("/Users/macbook/Data/sg historical rainfall")
def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "scripts").is_dir() and (candidate / "manifests").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from this notebook.")

REPO_ROOT = find_repo_root()

CREDIT_SNAPSHOT = {
    "credit_id": ALLOWED_CREDIT_ID,
    "status": "Available",
    "percent_remaining": 100,
    "remaining_usd": 50.00,
    "original_usd": 50.00,
    "snapshot_source": "user-provided",
    "snapshot_date": "2026-10-11",
}
COST_RESERVE_USD = 10.00
MAX_PLANNED_SPEND_USD = 40.00
CREDIT_LIVE_VERIFIED = False  # Refresh from the Education account Credits page first.
PLANNED_SESSION_COST_USD = None  # Use an estimate including pending and cleanup costs.

print(f"Notebook root: {REPO_ROOT}")
print(f"Project: {PROJECT_ID}")
print(f"Target billing: {BILLING_ACCOUNT_NAME} ({BILLING_ACCOUNT_ID})")
print(f"Allowed credit: {ALLOWED_CREDIT_ID}")
print("Credit snapshot: user-provided; refresh from Billing Console before any paid resource.")

Notebook root: /Users/macbook/Desktop/workspace/sg-rainfall-distributed-analysis-and-prediction
Project: new-cloud-256902
Target billing: Billing Account for Education (01DAD8-AE5F1A-E3910A)
Allowed credit: 3LU63BPGY77JFDQQ
Credit snapshot: user-provided; refresh from Billing Console before any paid resource.


In [2]:
REBUILD_MANIFEST = False  # Set True to rescan the eight local annual inputs.
manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
if REBUILD_MANIFEST:
    import importlib.util

    module_path = REPO_ROOT / "scripts" / "build_manifest.py"
    spec = importlib.util.spec_from_file_location("build_manifest", module_path)
    manifest_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(manifest_module)
    rebuilt = manifest_module.build_manifest(DATA_ROOT)
    temporary_path = manifest_path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(rebuilt, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(manifest_path)
    print(f"Rebuilt manifest: {manifest_path}")

if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_rows = [
    {
        "year": item["year"],
        "rows": item["row_count"],
        "size_bytes": item["size_bytes"],
        "sha256": item["sha256"],
        "schema_matches": item["schema_matches_expected"],
        "malformed_rows": item["rows_with_wrong_field_count"],
        "invalid_timestamps": item["unparseable_or_missing_timestamps"],
    }
    for item in manifest["files"]
]
display(manifest_rows)
print(f"Input years: {manifest['selection']['years']}")
print(f"Files in manifest: {len(manifest_rows)}")
print("Raw CSV files are read-only.")

[{'year': 2017,
  'rows': 5256106,
  'size_bytes': 881206891,
  'sha256': 'a51a27378607c6c37d51b72383259b27dae5dc7d5760a8b8fff04e86d9f92ddf',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2018,
  'rows': 4594246,
  'size_bytes': 769547920,
  'sha256': 'f42b40c37edbbc9a3ddb6e53e147606878280cbe1362e6580ea86aaf528db887',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2019,
  'rows': 5026862,
  'size_bytes': 842751786,
  'sha256': '1f30310b94912c800a96a9beedd65fcff62c663a1cbc2ff5606f0f174867d73c',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2020,
  'rows': 6334473,
  'size_bytes': 1048946924,
  'sha256': '2b8df4c6adebb3ea41183ca22fc261fdb34ef78a492bb205f660c431b924ec48',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2021,
  'rows': 7044646,
  'size_bytes': 1158573170,
  'sha256': '3bb97164ec298b4057303bfecea4acc0a824fc41d83aa3d

Input years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Files in manifest: 8
Raw CSV files are read-only.


In [3]:
def run_gcloud(args, *, json_output=False):
    result = subprocess.run(
        ["gcloud", *args],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or f"gcloud exited with {result.returncode}")
    if json_output:
        return json.loads(result.stdout or "null")
    if result.stdout.strip():
        print(result.stdout.strip())
    return result.stdout

def billing_project_state():
    return run_gcloud(
        ["billing", "projects", "describe", PROJECT_ID, "--format=json"],
        json_output=True,
    )

before = billing_project_state()
target = f"billingAccounts/{BILLING_ACCOUNT_ID}"
print({
    "project_id": PROJECT_ID,
    "billing_enabled_before": before.get("billingEnabled"),
    "already_linked_to_education": before.get("billingAccountName") == target,
})

{'project_id': 'new-cloud-256902', 'billing_enabled_before': False, 'already_linked_to_education': False}


In [4]:
LINK_PROJECT_TO_EDUCATION = True

if LINK_PROJECT_TO_EDUCATION:
    current = billing_project_state()
    if current.get("billingAccountName") == target and current.get("billingEnabled") is True:
        verified = current
        billing_ready = True
    else:
        try:
            run_gcloud([
                "billing", "projects", "link", PROJECT_ID,
                f"--billing-account={BILLING_ACCOUNT_ID}",
            ])
            verified = billing_project_state()
            billing_ready = (
                verified.get("billingAccountName") == target
                and verified.get("billingEnabled") is True
            )
        except RuntimeError as exc:
            message = str(exc)
            if "IAM_PERMISSION_DENIED" in message or "does not have permission" in message:
                print(
                    "Billing link blocked: the active gcloud identity lacks "
                    "project billing-assignment permissions. No bucket, cluster, or job was created."
                )
                verified = current
                billing_ready = False
            else:
                raise

    display({
        "project_id": PROJECT_ID,
        "billing_account": BILLING_ACCOUNT_NAME,
        "billing_account_id": BILLING_ACCOUNT_ID,
        "billing_enabled": verified.get("billingEnabled"),
        "linkage_verified": verified.get("billingAccountName") == target,
    })
else:
    billing_ready = False
    print("Skipped billing linkage. No cloud resources may be created.")

Billing link blocked: the active gcloud identity lacks project billing-assignment permissions. No bucket, cluster, or job was created.


{'project_id': 'new-cloud-256902',
 'billing_account': 'Billing Account for Education',
 'billing_account_id': '01DAD8-AE5F1A-E3910A',
 'billing_enabled': False,
 'linkage_verified': False}

In [5]:
credit_id_matches = CREDIT_SNAPSHOT["credit_id"] == ALLOWED_CREDIT_ID
credit_snapshot_available = (
    CREDIT_SNAPSHOT["status"] == "Available"
    and CREDIT_SNAPSHOT["remaining_usd"] > COST_RESERVE_USD
)

def require_cost_gate(estimate_usd):
    if not billing_ready:
        raise RuntimeError("Project is not linked to the permitted Education account.")
    if not CREDIT_LIVE_VERIFIED:
        raise RuntimeError("Refresh and verify this exact credit in Billing Console before paid work.")
    if not credit_id_matches or not credit_snapshot_available:
        raise RuntimeError("The sole permitted credit is missing, unavailable, or below its reserve.")
    if estimate_usd is None or estimate_usd < 0:
        raise RuntimeError("Enter a documented cost estimate before paid work.")
    allowed = min(MAX_PLANNED_SPEND_USD, CREDIT_SNAPSHOT["remaining_usd"] - COST_RESERVE_USD)
    if estimate_usd > allowed:
        raise RuntimeError(f"Estimate exceeds the allowed credit after reserve ({allowed:.2f} USD).")

display({
    "allowed_credit_id": ALLOWED_CREDIT_ID,
    "snapshot_credit_id_matches": credit_id_matches,
    "snapshot_remaining_usd": CREDIT_SNAPSHOT["remaining_usd"],
    "snapshot_is_live": CREDIT_LIVE_VERIFIED,
    "billing_project_ready": billing_ready,
    "paid_resource_gate": False,
})
print("No paid resource is permitted until the live credit snapshot and documented run estimate pass require_cost_gate(...).")

{'allowed_credit_id': '3LU63BPGY77JFDQQ',
 'snapshot_credit_id_matches': True,
 'snapshot_remaining_usd': 50.0,
 'snapshot_is_live': False,
 'billing_project_ready': False,
 'paid_resource_gate': False}

No paid resource is permitted until the live credit snapshot and documented run estimate pass require_cost_gate(...).


In [6]:
ENABLE_APIS = False
REQUIRED_APIS = [
    "dataproc.googleapis.com",
    "compute.googleapis.com",
    "storage.googleapis.com",
]

if ENABLE_APIS:
    if not billing_ready:
        raise RuntimeError("Billing preflight must pass before enabling APIs.")
    for api in REQUIRED_APIS:
        run_gcloud([
            "services", "enable", api,
            f"--project={PROJECT_ID}",
        ])
else:
    print("API enablement skipped; set ENABLE_APIS=True only after live credit and cost preflight.")

API enablement skipped; set ENABLE_APIS=True only after live credit and cost preflight.


In [7]:
CREATE_BUCKET = False
UPLOAD_INPUTS_AND_CODE = False

if CREATE_BUCKET:
    if not BUCKET:
        raise RuntimeError("Set a verified private bucket name first.")
    require_cost_gate(PLANNED_SESSION_COST_USD)
    run_gcloud([
        "storage", "buckets", "create", f"gs://{BUCKET}",
        f"--project={PROJECT_ID}",
        f"--location={REGION}",
        "--default-storage-class=STANDARD",
        "--uniform-bucket-level-access",
    ])
    print(f"Created private bucket gs://{BUCKET}")

if UPLOAD_INPUTS_AND_CODE:
    if not BUCKET:
        raise RuntimeError("Set a verified private bucket name first.")
    require_cost_gate(PLANNED_SESSION_COST_USD)
    manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    run_gcloud(["storage", "cp", str(manifest_path), f"gs://{BUCKET}/manifests/"])
    for year in range(2017, 2025):
        source = DATA_ROOT / "raw" / f"HistoricalRainfallacrossSingapore{year}.csv"
        if not source.is_file():
            raise FileNotFoundError(source)
        run_gcloud(["storage", "cp", str(source), f"gs://{BUCKET}/raw/"])
    for source in [
        REPO_ROOT / "scripts" / "run_quality_audit.py",
        REPO_ROOT / "scripts" / "build_manifest.py",
        REPO_ROOT / "notebooks" / "01_data_quality.ipynb",
    ]:
        run_gcloud(["storage", "cp", str(source), f"gs://{BUCKET}/code/"])
    print(f"Uploaded manifest, eight selected CSVs, and notebook code to gs://{BUCKET}/")
else:
    print("Bucket creation and uploads skipped; both are cost-bearing and require a refreshed cost gate.")

Bucket creation and uploads skipped; both are cost-bearing and require a refreshed cost gate.


In [8]:
CREATE_CLUSTER = False
DATAPROC_IMAGE = ""  # Pin an exact supported image after a no-cost compatibility review.
SUBNET = ""
VM_SERVICE_ACCOUNT = ""
COST_ESTIMATE_USD = PLANNED_SESSION_COST_USD

if CREATE_CLUSTER:
    if not BUCKET or not SUBNET or not VM_SERVICE_ACCOUNT or not DATAPROC_IMAGE:
        raise RuntimeError("Set bucket, subnet, service account, and exact image first.")
    require_cost_gate(COST_ESTIMATE_USD)
    run_gcloud([
        "dataproc", "clusters", "create", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}",
        f"--image-version={DATAPROC_IMAGE}",
        f"--subnet={SUBNET}", "--no-address",
        f"--service-account={VM_SERVICE_ACCOUNT}", "--scopes=cloud-platform",
        "--master-machine-type=e2-standard-4",
        "--master-boot-disk-type=pd-balanced", "--master-boot-disk-size=100GB",
        "--num-workers=2", "--worker-machine-type=e2-standard-4",
        "--worker-boot-disk-type=pd-balanced", "--worker-boot-disk-size=100GB",
        f"--bucket={BUCKET}", f"--temp-bucket={BUCKET}",
        "--enable-component-gateway", "--optional-components=JUPYTER",
        f"--properties=dataproc:jupyter.notebook.gcs.dir=gs://{BUCKET}/notebooks/,spark:spark.eventLog.enabled=true,spark:spark.eventLog.dir=gs://{BUCKET}/spark-history/",
        "--delete-max-idle=45m", "--delete-max-age=3h",
    ])
    print(f"Created Dataproc cluster {CLUSTER} in {REGION}.")
else:
    print("Dataproc creation skipped. Complete image, network, identity, and cost checks first.")

Dataproc creation skipped. Complete image, network, identity, and cost checks first.


In [9]:
DELETE_CLUSTER = False

if DELETE_CLUSTER:
    run_gcloud([
        "dataproc", "clusters", "delete", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}", "--quiet",
    ])
    remaining = run_gcloud([
        "dataproc", "clusters", "list",
        f"--project={PROJECT_ID}", f"--region={REGION}",
        "--format=json",
    ], json_output=True)
    display({"remaining_cluster_names": [item["clusterName"] for item in remaining]})
else:
    print("Cluster deletion skipped. Set DELETE_CLUSTER=True after outputs and notebooks are persisted.")

Cluster deletion skipped. Set DELETE_CLUSTER=True after outputs and notebooks are persisted.


## Spark Web UI evidence

After each Spark action in `01_data_quality.ipynb` or a later analysis notebook finishes, open the corresponding **Spark Web UI**. Review the Jobs summary and every Job detail page, then save screenshots under `report/evidence/spark-ui/<RUN_ID>/`. Use Spark History Server for a completed application. Do not substitute Dataproc Jobs or YARN screenshots.